# 03 — FBref ↔ Understat Matching & Cleaning (v2)

Rebuilt to use the new combined FBref source (`observation_scope` from the Data Integration notebook) instead of the original per-season FBref pull. This source already includes progressive passes/carries, key passes, tackles, interceptions, aerial duels, and FBref's own xG/xAG — the features the original pipeline was missing.

Understat is still merged in separately for `xGChain`/`xGBuildup` (buildup-involvement metrics FBref doesn't have) and as a cross-check on `xG`/`xA`.

**Lessons folded in from the previous version of this notebook:**
- Names are accent-stripped from the very first `player_key` built (the earlier version's bug — accented names like 'Mbappé' scoring too low on fuzzy matching and being silently dropped — is avoided by never creating an un-normalized `player_key` in the first place).
- Multi-club players are resolved to their **primary club** (most minutes) rather than dropped or left duplicated, per the agreed decision.
- FBref's `Min` is treated as the single source of truth for the 900-minute threshold.

> **Portfolio reproducibility note:** This notebook requires the filtered FBref observation-scope CSV from Notebook 01 and the combined Understat workbook from Notebook 02. These source files were not present in the supplied project archive, so the cleaned notebook has not been claimed as end-to-end executed. Both upstream notebooks write these files to `data/processed/`; Notebook 03 reads them from there. The notebook preserves the original matching thresholds and manual exclusions.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import html
import unicodedata
from rapidfuzz import process, fuzz, utils

# Resolve the repository root whether this notebook is run from the root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Notebooks 01 and 02 both write their outputs to data/processed/.
FBREF_FILE = PROCESSED_DIR / 'Top5_League_Players_observation_scope.csv'
UNDERSTAT_FILE = PROCESSED_DIR / 'Understat_dataset.xlsx'

missing = [str(p) for p in (FBREF_FILE, UNDERSTAT_FILE) if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Required source file(s) not found. See data/README.md for the expected files:\n' + '\n'.join(missing)
    )


## Environment

Install the project dependencies from the repository root with `pip install -r requirements.txt`. This notebook uses RapidFuzz for approximate name matching; it does not install packages from inside the notebook.

In [2]:
def normalize_name(name):
    """Decode HTML entities, remove accents and standardise player names."""
    name = html.unescape(str(name))
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    return name.strip().lower()


## 1. Name normalization (accent-safe from the start)

In [3]:
def normalize_name(name):
    """Strip accents, decode HTML entities, lowercase, trim whitespace."""
    name = html.unescape(str(name))
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    return name.strip().lower()

## 2. Load FBref (new combined source) and apply the minutes threshold

`observation_scope` from the integration notebook already filters to Big 5 leagues, 2019-20 to 2023-24, and age 18-24 — the 900-minute threshold is applied here, using `Min` as the agreed source of truth.

In [4]:
df_fbref = pd.read_csv(FBREF_FILE)

# Notebook 01 exports the season field as lowercase 'season'.
# Standardise it here because the matching workflow uses 'Season'.
if 'Season' not in df_fbref.columns and 'season' in df_fbref.columns:
    df_fbref = df_fbref.rename(columns={'season': 'Season'})

print(f'Rows before minutes filter: {len(df_fbref):,}')
df_fbref = df_fbref[df_fbref['Min'] >= 900].copy()
print(f'Rows after 900+ minute filter: {len(df_fbref):,}')

df_fbref['player_key'] = df_fbref['Player'].apply(normalize_name)
df_fbref['season_clean'] = df_fbref['Season'].astype(str).str.replace(' ', '', regex=False)


Rows before minutes filter: 6,231
Rows after 900+ minute filter: 3,023


## 3. Resolve multi-club player-seasons to their primary club

If a player has more than one row for the same `player_key` + `Season` (a mid-season transfer, however the source represents it), keep only the row with the most minutes — the club representing the majority of their season.

In [5]:
dupe_counts = df_fbref.groupby(['player_key', 'season_clean']).size()
multi_club_keys = dupe_counts[dupe_counts > 1]
print(f"Player-seasons with multiple club rows: {len(multi_club_keys)}")

df_fbref = df_fbref.sort_values('Min', ascending=False).drop_duplicates(
    subset=['player_key', 'season_clean'], keep='first'
).reset_index(drop=True)

print(f"Rows after keeping primary club only: {len(df_fbref)}")

Player-seasons with multiple club rows: 16
Rows after keeping primary club only: 3007


## 4. Load Understat

Same source and cleaning as before — HTML-entity decoding, accent-safe `player_key`, and season-label standardisation to match FBref's format.

In [6]:
df_understat = pd.read_excel(UNDERSTAT_FILE)

df_understat['season_clean'] = df_understat['Season'].astype(str).str.replace(' ', '', regex=False)
df_understat['player_key'] = df_understat['player'].apply(normalize_name)

print(f'Understat rows: {len(df_understat):,}')
print(f'Understat seasons: {sorted(df_understat["season_clean"].dropna().unique())}')


Understat rows: 18,827
Understat seasons: ['2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']


In [7]:
df_understat = df_understat.sort_values('min', ascending=False).drop_duplicates(
    subset=['player_key', 'season_clean'], keep='first'
).reset_index(drop=True)

## 5. Exact match on player_key + season

In [8]:
df_merged = pd.merge(
    df_fbref,
    df_understat,
    on=['player_key', 'season_clean'],
    how='inner',
    suffixes=('', '_understat'),
)

print(f"Exact matches: {len(df_merged)} / {len(df_fbref)} FBref rows "
      f"({len(df_merged) / len(df_fbref):.1%})")

Exact matches: 2654 / 3007 FBref rows (88.3%)


## 6. Fuzzy-match the leftovers, blocked by team + season

In [9]:
matched_keys = set(zip(df_merged['player_key'], df_merged['season_clean']))

unmatched_fbref = df_fbref[
    ~df_fbref.apply(lambda r: (r['player_key'], r['season_clean']) in matched_keys, axis=1)
].copy()

print(f"Unmatched (candidates for fuzzy matching): {len(unmatched_fbref)}")

Unmatched (candidates for fuzzy matching): 353


In [10]:
# Team blocking removed — diagnostic testing showed it was failing even for
# unambiguous clubs (Arsenal, Barcelona, Bayern Munich all showed up as
# 'unmapped' despite identical spelling in both sources), and relaxing to
# season-only blocking recovered genuine matches (e.g. Rayan Ait-Nouri at
# 100%) that team blocking had incorrectly excluded. Season alone is a
# strong enough constraint here (a few hundred candidates, not the full
# dataset) for rapidfuzz to discriminate reliably on full names.
HIGH_THRESHOLD = 90
REVIEW_THRESHOLD = 75

auto_rows, review_rows, still_unmatched_rows = [], [], []

for _, row in unmatched_fbref.iterrows():
    season = row['season_clean']
    target_name = row['player_key']

    candidates = df_understat[df_understat['season_clean'] == season]

    if candidates.empty:
        still_unmatched_rows.append(row)
        continue

    candidate_names = candidates['player_key'].tolist()
    best = process.extractOne(target_name, candidate_names,
                               scorer=fuzz.token_sort_ratio, processor=utils.default_process)

    if best is None:
        still_unmatched_rows.append(row)
        continue

    matched_name, score, idx = best
    result_row = row.copy()
    result_row['matched_understat_player'] = candidates.iloc[idx]['player']
    result_row['match_score'] = score

    if score >= HIGH_THRESHOLD:
        auto_rows.append(result_row)
    elif score >= REVIEW_THRESHOLD:
        review_rows.append(result_row)
    else:
        still_unmatched_rows.append(row)

auto_matches = pd.DataFrame(auto_rows)
review_matches = pd.DataFrame(review_rows)
still_unmatched = pd.DataFrame(still_unmatched_rows)

print(f"Auto-matched (score >= {HIGH_THRESHOLD}): {len(auto_matches)}")
print(f"Needs manual review ({REVIEW_THRESHOLD}-{HIGH_THRESHOLD}): {len(review_matches)}")
print(f"Still unmatched (< {REVIEW_THRESHOLD}): {len(still_unmatched)}")

Auto-matched (score >= 90): 84
Needs manual review (75-90): 156
Still unmatched (< 75): 113


### Eyeball the review tier before accepting

Set `pd.set_option('display.max_rows', None)` if the table is long.

In [11]:
pd.set_option('display.max_rows', None)
review_matches[['Player', 'matched_understat_player', 'match_score', 'Team', 'Season']].sort_values('match_score', ascending=False)

,Player,matched_understat_player,match_score,Team,Season
2682,Zaydou Youssouf,Zaydou Youssef,89.655172,Saint-Étienne,2019-2020
1889,Zaydou Youssouf,Zaydou Youssef,89.655172,Saint-Étienne,2021-2022
2457,Zaydou Youssouf,Zaydou Youssef,89.655172,Saint-Étienne,2020-2021
240,Roger Ibanez,Robert Ibáñez,88.000000,Roma,2021-2022
1064,Junior Dina Ebimbe,Eric Junior Dina Ebimbe,87.804878,Dijon,2020-2021
2705,Junior Dina Ebimbe,Eric Junior Dina Ebimbe,87.804878,Eint Frankfurt,2022-2023
1329,Junior Dina Ebimbe,Eric Junior Dina Ebimbe,87.804878,Eint Frankfurt,2023-2024
2897,Nayef Aguerd,Naif Aguerd,86.956522,Dijon,2019-2020
77,Nayef Aguerd,Naif Aguerd,86.956522,Rennes,2020-2021
2383,Luca Waldschmidt,Gian-Luca Waldschmidt,86.486486,Freiburg,2019-2020


In [12]:
discard_pairs = [
    ('Marvin', 'Mariano'),
    ('Muhammed Cham	Muhammed Damar'),
    ('Ibrahim Sulemana', 'Ibrahim Sangare'),
    ('Alejandro Balde', 'Alejandro Catena'),
    ('Roger Ibanez', 'Roger Martínez'),
    ('Matìas Soulé', 'Mathias Olesen'),
    ('Martin Ødegaard', 'Aarón Martín'),
    ('Lucas Beraldo', 'Lucas Beltrán'),
    ('Marlon', 'Marcelo'),
    ('Jonathan David', 'Jonathan Gradit'),
    ('Igor', 'Iago'),
    ('Dani Ceballos,	Dani Calvo'),
    ('Wesley Moraes',	'Wesley Moustache'),
]

confirmed_review = review_matches[
    ~review_matches.apply(lambda r: (r['Player'], r['matched_understat_player']) in discard_pairs, axis=1)
]

print(f"Discarding: {len(discard_pairs)} pairs")
print(f"Confirmed: {len(confirmed_review)} / {len(review_matches)} review-tier rows")

Discarding: 13 pairs
Confirmed: 134 / 156 review-tier rows


## 7. Attach Understat stats to accepted fuzzy matches, combine with exact matches

In [13]:
understat_stats_cols = [c for c in df_understat.columns
                        if c not in ('player', 'team', 'Season', 'season_clean', 'player_key', '_team_key')]

understat_lookup = df_understat.copy()
understat_lookup['matched_understat_player'] = understat_lookup['player']

accepted = pd.concat([auto_matches, confirmed_review], ignore_index=True)

fuzzy_merged = accepted.merge(
    understat_lookup[['matched_understat_player', 'season_clean'] + understat_stats_cols],
    on=['matched_understat_player', 'season_clean'], how='left'
)
fuzzy_merged_clean = fuzzy_merged.drop(
    columns=['_team_key', 'match_score', 'matched_understat_player'], errors='ignore'
)

df_merged_final = pd.concat([df_merged, fuzzy_merged_clean], ignore_index=True)

print(f"df_merged (exact only): {len(df_merged)}")
print(f"df_merged_final (exact + fuzzy): {len(df_merged_final)}")

df_merged (exact only): 2654
df_merged_final (exact + fuzzy): 2872


### Check your named case-study players directly

In [14]:
case_study_players = ['Cole Palmer', 'Michael Olise', 'Morgan Rogers', 'Viktor Gyökeres']
for name in case_study_players:
    key = normalize_name(name)
    rows = df_merged_final[df_merged_final['player_key'] == key]
    print(f"\n{name}: {len(rows)} matched season(s)")
    if not rows.empty:
        print(rows[['Season', 'Team']].to_string(index=False))


Cole Palmer: 1 matched season(s)
   Season    Team
2023-2024 Chelsea

Michael Olise: 3 matched season(s)
   Season           Team
2022-2023 Crystal Palace
2023-2024 Crystal Palace
2021-2022 Crystal Palace

Morgan Rogers: 0 matched season(s)

Viktor Gyökeres: 0 matched season(s)


## 8. Clean up helper columns and save

In [15]:
df_merged_final = df_merged_final.drop(
    columns=['season_clean', '_team_key', '_season_end_year'], errors='ignore'
)

output_file = PROCESSED_DIR / 'combined_fbref_understat_final.csv'
df_merged_final.to_csv(output_file, index=False)
print(f'Saved: {output_file.relative_to(PROJECT_ROOT)} ({df_merged_final.shape})')


Saved: data/processed/combined_fbref_understat_final.csv ((2872, 286))


## 9. Match-rate summary for your data section

In [16]:
total_fbref_rows = len(df_fbref)
exact_matched = len(df_merged)
fuzzy_matched = len(accepted)
dropped = len(still_unmatched)

print(f"Total FBref player-seasons (post minutes/primary-club filtering): {total_fbref_rows}")
print(f"Exact matches: {exact_matched} ({exact_matched/total_fbref_rows:.1%})")
print(f"Fuzzy matches accepted: {fuzzy_matched} ({fuzzy_matched/total_fbref_rows:.1%})")
print(f"Dropped as unresolvable: {dropped} ({dropped/total_fbref_rows:.1%})")
print(f"Overall match rate: {(exact_matched+fuzzy_matched)/total_fbref_rows:.1%}")

Total FBref player-seasons (post minutes/primary-club filtering): 3007
Exact matches: 2654 (88.3%)
Fuzzy matches accepted: 218 (7.2%)
Dropped as unresolvable: 113 (3.8%)
Overall match rate: 95.5%


## Optional diagnostic: investigate unmatched players

The original research notebook included a surname-based diagnostic to inspect a sample of unmatched players. This is exploratory only: a surname appearing elsewhere in Understat does not establish that two records represent the same person. For portfolio use, unmatched records should remain unmatched unless independently verified.

## Matching identifiers

The original notebook contained a later diagnostic that generated synthetic row identifiers (`fbref_...` and `understat_...`). These are not genuine provider IDs and do not establish that two records belong to the same player. That diagnostic has therefore been removed from the cleaned workflow.

In [17]:
dupe_understat = df_understat.groupby(['player_key', 'season_clean']).size()
multi_understat = dupe_understat[dupe_understat > 1]
print(f"Understat player-seasons with multiple rows: {len(multi_understat)}")

Understat player-seasons with multiple rows: 0
